# Day 3 — Train BaselineCNN from scratch

Run this on **Kaggle** with:
- the `emmarex/plantdisease` dataset added (Add Input -> search `plantdisease`),
- **Accelerator: GPU T4 x2** (or P100) — on CPU 10 epochs would take hours,
- **Internet on**, so the cell below can clone the `leafguard` repo.

Goal: train `BaselineCNN` for 10 epochs on the seeded 70/15/15 train/val/test split. The best checkpoint is
picked on **val accuracy only**; the test split is held out untouched for Day 4 evaluation (its seed, sizes and
indices are saved in `metrics_baseline.json`).

Outputs land in `/kaggle/working/outputs/`: `baseline_best.pt`, `metrics_baseline.json`, `training_curves_baseline.png`.

In [ ]:
import os

# Kaggle has used both mount layouts; Day 1 found the second one.
CANDIDATE_ROOTS = [
    "/kaggle/input/plantdisease/PlantVillage",
    "/kaggle/input/datasets/emmarex/plantdisease/PlantVillage",
]
DATA_ROOT = next((p for p in CANDIDATE_ROOTS if os.path.isdir(p)), None)
assert DATA_ROOT, f"PlantVillage not found in any of {CANDIDATE_ROOTS} -- is the dataset added?"
print("DATA_ROOT:", DATA_ROOT)

In [ ]:
import inspect
import subprocess
import sys

# Use an uploaded copy of src/ if there is one; otherwise clone the repo (needs Internet on).
# The Day 3 code must be on this ref: if the Day 3 PR isn't merged yet, use "day-3-baseline-training".
LEAFGUARD_REF = "main"
CLONE_DIR = "/kaggle/working/leafguard"

for path in ["/kaggle/input/leafguard-src", f"{CLONE_DIR}/src"]:
    if os.path.isdir(path) and path not in sys.path:
        sys.path.insert(0, path)

try:
    import leafguard  # noqa: F401
except ImportError:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", LEAFGUARD_REF,
         "https://github.com/Bloop31/leafguard.git", CLONE_DIR],
        check=True,
    )
    sys.path.insert(0, f"{CLONE_DIR}/src")
    import leafguard  # noqa: F401

from leafguard.train import train

assert "seed" in inspect.signature(train).parameters, (
    f"{leafguard.__file__} is pre-Day-3 code (no seed / test split). "
    "Set LEAFGUARD_REF to a ref that has the Day 3 changes, delete /kaggle/working/leafguard and re-run."
)
print("leafguard", leafguard.__version__, "from", leafguard.__file__)

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU -- set Accelerator to GPU in the notebook settings."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import time

OUT_DIR = "/kaggle/working/outputs"
BACKBONE = "baseline"

t0 = time.time()
metrics = train(
    backbone=BACKBONE,
    data_root=DATA_ROOT,
    out_dir=OUT_DIR,
    epochs=10,
    batch_size=32,
    lr=1e-3,
    seed=42,
)
print(f"Total training time: {(time.time() - t0) / 60:.1f} min")

In [ ]:
from IPython.display import Image, display

display(Image(filename=f"{OUT_DIR}/training_curves_{BACKBONE}.png"))

In [ ]:
h = metrics["history"]
best = metrics["best_epoch"] - 1
split = metrics["split"]

print(f"Best val acc:          {metrics['best_val_acc']:.4f} (epoch {metrics['best_epoch']} of {len(h['val_acc'])})")
print(f"Train acc at best:     {h['train_acc'][best]:.4f}  -> train-val gap {h['train_acc'][best] - h['val_acc'][best]:+.4f}")
print(f"Final epoch train/val: {h['train_acc'][-1]:.4f} / {h['val_acc'][-1]:.4f}  "
      f"-> gap {h['train_acc'][-1] - h['val_acc'][-1]:+.4f}")
print(f"Lowest val loss:       {min(h['val_loss']):.4f} (epoch {h['val_loss'].index(min(h['val_loss'])) + 1})")
print(f"Mean time per epoch:   {sum(h['epoch_time_s']) / len(h['epoch_time_s']):.1f}s")
print(f"Split (seed={split['seed']}): train={split['n_train']} val={split['n_val']} "
      f"test={split['n_test']} (held out, not evaluated)")
print()
print("Files to download from", OUT_DIR)
for name in sorted(os.listdir(OUT_DIR)):
    print(f"  {name:<32} {os.path.getsize(os.path.join(OUT_DIR, name)) / 1e6:.2f} MB")

## Findings (fill this in after running)

- Best val acc (and which epoch):
- Epochs trained / time per epoch:
- Train vs val accuracy gap at the best epoch:
- Overfitting? (val loss rising while train loss keeps falling?)
- Anything unexpected (slow data loading, unstable loss, GPU memory)?